# PROTOTYPE — Interactive docking box wireframe (DDOS-6937)

**Throwaway.** Validates: molstar viewer + Apply → Python receives `rotation_deg`.

1. Run cell 1 — viewer loads with BRD4 protein + docking box.
2. Rotate via Settings → Docking Box (gear icon) and/or overlay sliders.
3. Click **Apply to notebook** — cell 2 prints the committed dict.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent

PROTO_DIR = REPO_ROOT / "prototypes" / "ddos-6937-interactive-box-wireframe"
sys.path.insert(0, str(PROTO_DIR))

from comm_bridge import render_interactive_html_with_comm
from interactive_box_html import render_interactive_docking_box_html

from deeporigin.drug_discovery import Pocket
from deeporigin.drug_discovery.docking_common import resolve_docking_box_geometry

pocket_fixture = (
    REPO_ROOT / "tests" / "fixtures" / "files" / "pocketfinder" / "pocket_1.pdb"
)
brd_pdb = REPO_ROOT / "src" / "data" / "brd" / "brd.pdb"

pocket = Pocket.from_pdb_file(str(pocket_fixture), name="pocket-1")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 15.0
box_center, box_size = resolve_docking_box_geometry(pocket)

committed_state: dict | None = None


def on_commit(payload: dict) -> None:
    global committed_state
    committed_state = payload
    print("\n=== Committed docking box ===")
    for key, value in payload.items():
        print(f"  {key}: {value}")


handle = render_interactive_html_with_comm(
    lambda bridge_id: render_interactive_docking_box_html(
        pdb_path=str(brd_pdb),
        box_center=list(box_center),
        box_size=list(box_size),
        bridge_id=bridge_id,
    ),
    on_commit=on_commit,
)

print(f"Bridge id: {handle.bridge_id}")
print(f"Box center: {list(box_center)}")
print(f"Box size: {list(box_size)}")

In [ ]:
# Re-run after Apply to inspect the last committed payload.
if committed_state is None:
    print("No commit yet — click Apply to notebook in the viewer.")
else:
    print(committed_state)